# Huấn luyện Plant Disease AI — MobileNetV2 (15 lớp)

Notebook tự động tải PlantVillage bằng TensorFlow Datasets, lọc đúng 15 lớp, xuất ảnh theo cấu trúc thư mục, tiền xử lý, chia train/validation/test, huấn luyện MobileNetV2 pretrained ImageNet và đánh giá trên tập test độc lập.

**Bạn không cần tải dataset lên Drive hoặc máy tính.** Lần đầu chạy cần Internet và dung lượng Colab đủ lớn. Model cũ không bị ghi đè.

In [ ]:
# 1. Cài thư viện và kết nối Google Drive
!pip -q install tensorflow-datasets scikit-learn matplotlib
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, json, shutil, re
from pathlib import Path
import tensorflow as tf
import tensorflow_datasets as tfds
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "Không có GPU — huấn luyện sẽ chậm")

In [ ]:
# 2. Lấy mã nguồn đúng branch của PR
REPO_DIR = "/content/Plant-Disease-Detection"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone","--branch","feature/reproducible-mobilenetv2-colab-training","https://github.com/ShimeKano/Plant-Disease-Detection.git",REPO_DIR], check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"pull"],check=True)
print("Source:", REPO_DIR)

In [ ]:
# 3. Tải PlantVillage tự động và lọc đúng 15 lớp
# TensorFlow Datasets tải dữ liệu gốc vào cache Colab, không cần bạn upload ZIP.
EXPECTED = [
 "Pepper__bell___Bacterial_spot","Pepper__bell___healthy",
 "Potato___Early_blight","Potato___Late_blight","Potato___healthy",
 "Tomato__Bacterial_spot","Tomato__Early_blight","Tomato__Late_blight",
 "Tomato__Leaf_Mold","Tomato__Septoria_leaf_spot",
 "Tomato__Spider_mites_Two_spotted_spider_mite","Tomato__Target_Spot",
 "Tomato__Tomato_YellowLeaf__Curl_Virus","Tomato__Tomato_mosaic_virus","Tomato__healthy"
]
def norm_label(s): return re.sub(r"[^a-z0-9]", "", s.lower())
# Tên nhãn TFDS có dấu phẩy/underscore khác tên thư mục trong app;
# chuẩn hóa để đối chiếu mà không phụ thuộc số lượng dấu gạch dưới.
label_names = tfds.builder("plant_village").info.features["label"].names
lookup = {norm_label(n): n for n in label_names}
label_to_folder = {}
for folder in EXPECTED:
    key = norm_label(folder)
    if key not in lookup:
        raise ValueError(f"Không tìm thấy lớp {folder} trong TFDS. Nhãn có sẵn: {label_names}")
    label_to_folder[lookup[key]] = folder
print("15 lớp được chọn:", len(label_to_folder))
print("Đang tải PlantVillage (lần đầu có thể mất vài phút)...")
ds, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True, shuffle_files=False)
DATASET_DIR = Path("/content/plantvillage_15")
if DATASET_DIR.exists(): shutil.rmtree(DATASET_DIR)
for folder in EXPECTED: (DATASET_DIR/folder).mkdir(parents=True, exist_ok=True)
counts = {name: 0 for name in EXPECTED}
for image, label in tfds.as_numpy(ds):
    original = info.features["label"].int2str(int(label))
    folder = label_to_folder.get(original)
    if folder is None: continue
    out = DATASET_DIR/folder/f"{counts[folder]:06d}.jpg"
    tf.io.write_file(str(out), tf.io.encode_jpeg(image, quality=95))
    counts[folder] += 1
print("Hoàn tất tải/lọc dữ liệu:")
for k,v in counts.items(): print(k, v)
missing = [k for k,v in counts.items() if v < 3]
if missing: raise RuntimeError(f"Lớp thiếu ảnh hoặc không khớp nhãn TFDS: {missing}")
print("Tổng ảnh:", sum(counts.values()))

In [ ]:
# 4. Cấu hình huấn luyện
OUTPUT_DIR = "/content/plant_disease_training_output"
IMG_SIZE = 128
BATCH_SIZE = 32
HEAD_EPOCHS = 12
FINE_TUNE_EPOCHS = 8
SEED = 42
print("Dataset:", DATASET_DIR, "| Output:", OUTPUT_DIR)

In [ ]:
# 5. Huấn luyện MobileNetV2 + tiền xử lý + chia tập + đánh giá
command = [sys.executable, f"{REPO_DIR}/src/train.py",
 "--data-dir", str(DATASET_DIR), "--output-dir", OUTPUT_DIR,
 "--img-size", str(IMG_SIZE), "--batch-size", str(BATCH_SIZE),
 "--epochs", str(HEAD_EPOCHS), "--fine-tune-epochs", str(FINE_TUNE_EPOCHS), "--seed", str(SEED)]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
# 6. Đóng gói model và báo cáo vào Google Drive
out = Path(OUTPUT_DIR)
required = ["best_model.keras","class_names.json","training_metadata.json","classification_report.txt","classification_report.json","confusion_matrix.png","learning_curves.png"]
missing = [name for name in required if not (out/name).exists()]
if missing: raise FileNotFoundError(f"Thiếu artifacts: {missing}")
archive = shutil.make_archive("/content/plant_disease_training_artifacts", "zip", OUTPUT_DIR)
destination = "/content/drive/MyDrive/plant_disease_training_artifacts.zip"
shutil.copy2(archive, destination)
print("Đã lưu ZIP:", destination)
print("Trong ZIP có best_model.keras — đây là file model đã train.")
from google.colab import files
files.download(archive)

## Sau khi chạy xong

- Tải về file `plant_disease_training_artifacts.zip` hoặc lấy từ Google Drive.
- File model chính: `best_model.keras`.
- Kèm theo: `class_names.json`, metadata, báo cáo từng lớp, confusion matrix và learning curves.
- Notebook **không tự đẩy model lên GitHub** và không ghi đè model hiện tại.
- Chỉ dùng các chỉ số thực tế được sinh ra sau khi chạy xong; PlantVillage là ảnh chụp điều kiện kiểm soát nên kết quả không đảm bảo tương đương ảnh ngoài đồng.